In [1]:
3+3

6

In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage,ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field
load_dotenv()

True

In [3]:
llm=ChatOpenAI(model='gpt-3.5-turbo')
message=' In 1 sentence, what does it mean for an AI agent to be autonomous'
reply=llm.invoke(message)
print(reply.content)

It means the AI agent can operate and make decisions independently without the need for constant human intervention.


In [11]:
for chunk in llm.stream('tell me a too line poem about autonomous agents'):
    print(chunk.content, end="")

Autonomous agents, programmed to roam,
In a world of their own, they make technology home.

In [ ]:
message = [SystemMessage('you are a assistant who answer is exactly five words'),
           HumanMessage("What is the capital of France")]

print(llm.invoke(message).content)



Paris is the capital city.


In [13]:
messages_as_dict =[
    {"role":"system", "content":"you are a assistant who answer in five words"},
    {"role":"user", "content":"what is the capital of France"}
]

print(llm.invoke(messages_as_dict).content)

Paris is the capital city.


In [14]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticket symbol"""
    fake_prices = {"AAPL": 241.2,"GOOG": 168.3,"AMZN":198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print("name:", get_share_price.name)
print("description", get_share_price.description)
print("args:", get_share_price.args)
print("called directly",get_share_price.invoke({"symbol":"AAPL"}))

name: get_share_price
description Return the current share price for a given ticket symbol
args: {'symbol': {'title': 'Symbol', 'type': 'string'}}
called directly 241.2


In [17]:
llm_with_tools=llm.bind_tools([get_share_price])
response=llm_with_tools.invoke("what is the share price of Amazon")
response.content
print("tool_calls:", response.tool_calls)

tool_calls: [{'name': 'get_share_price', 'args': {'symbol': 'AMZN'}, 'id': 'call_eHgEnt728ueVe6hta7ei9uUN', 'type': 'tool_call'}]


In [18]:
conversation =[HumanMessage('what is the share price of Amazon?')]
ai_message = llm_with_tools.invoke(conversation)
conversation.append(ai_message)

for call in ai_message.tool_calls:
    if call['name'] == "get_share_price":
        result = get_share_price.invoke(call['args'])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call['id']))

final = llm_with_tools.invoke(conversation)
print(final.content)

The share price of Amazon is $198.00.


In [19]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticket: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm.with_structured_output(Company)
company = structured_llm.invoke("Tell me about amazon the technology company")
print(company)
print("Just the ticket:", company.ticket)

c:\Users\a810557\akash\LangchainUpdated\.venv\Lib\site-packages\langchain_openai\chat_models\base.py:2480: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


name='Amazon' ticket='AMZN' founded_year=1994
Just the ticket: AMZN
